# BWE Stufe 1 — Regressions-Training (Kaggle/GPU)

**Auf Kaggle ausführen** (GPU-Notebook, P100/T4). Vorbereitung:
1. Settings → Accelerator → **GPU**.
2. **Add Data** → MUSDB18-HQ-Dataset einbinden (z. B. `quanglvitlm/musdb18-hq`).
3. Privates GitHub-Repo: Add-ons → **Secrets** → `GITHUB_TOKEN` (sonst öffentlich klonen).

Lokal nicht ausführbar (klont Repo, braucht GPU + Datensatz).

In [ ]:
# GPU prüfen
import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Code holen & installieren

TF ist auf Kaggle vorinstalliert → `--no-deps`, damit die GPU-Build nicht ersetzt wird; nur die Audio-Pakete nachinstallieren.

In [ ]:
REPO = "https://github.com/DanyelRychter/bwe_dnn.git"   # ggf. anpassen
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass  # öffentliches Repo

import subprocess
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/bwe_dnn"], check=True)
%pip install -q -e /kaggle/working/bwe_dnn --no-deps
%pip install -q librosa soundfile soxr

## 2. Pfade setzen (VOR dem bwe-Import!)

`BWE_DATA_ROOT` = Ordner, der `train/` und `test/` enthält. Ein evtl. vorhandener `val/`-Ordner wird ignoriert (kanonischer Split via Track-Namen).

In [ ]:
import os
# >>> An die tatsächliche Dataset-Struktur anpassen: <<<
# os.environ["BWE_DATA_ROOT"] = "/kaggle/input/musdb18-hq"     # muss train/ und test/ enthalten
os.environ["BWE_DATA_ROOT"] = "/kaggle/input/datasets/quanglvitlm/musdb18-hq"     # muss train/ und test/ enthalten
os.environ["BWE_CKPT_ROOT"] = "/kaggle/working/bwe_runs"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import sys; sys.path.insert(0, "/kaggle/working/bwe_dnn")
from bwe import config as cfg
print(cfg.summary())

In [ ]:
# Datensatz-Check: erwartete 86/14/50 (bzw. tatsächliche Zahlen)
from bwe.data import splits as SP
for s in SP.SPLIT_NAMES:
    print(f"{s:6s}: {len(SP.get_split(s)):3d} Tracks")

## 3. Subset-Training (Architektur/Hyperparameter)

Wenige Tracks → schnell prüfen, dass Val-LSD-HF unter die Copy-Up-Baseline sinkt.

In [ ]:
from bwe.train.regression import train
model, hist = train(run="reg_subset", mode="subset", batch_size=16, epochs=30, limit=20)

## 4. Volltraining

Voller Datensatz, resumebar (BackupAndRestore). Bricht eine Session ab (12-h-Grenze), Zelle einfach erneut ausführen.

In [ ]:
from bwe.train.regression import train
model, hist = train(run="reg_full", mode="full", batch_size=16, epochs=cfg.EPOCHS)

## 5. Lernkurve + Persistenz

Gewichte unter `/kaggle/working/bwe_runs/reg_full/` (best.weights.h5 + generator.weights.h5).
Für Phase D / das Ergebnis-Notebook herunterladen oder als Kaggle-Output-Dataset sichern.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("/kaggle/working/bwe_runs/reg_full/log.csv")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(log["epoch"], log["loss"], label="train"); ax[0].plot(log["epoch"], log["val_loss"], label="val")
ax[0].set_title("RI+Mag-Loss"); ax[0].set_xlabel("Epoche"); ax[0].legend()
ax[1].plot(log["epoch"], log["val_lsd_hf"]); ax[1].set_title("Val-LSD-HF [dB]"); ax[1].set_xlabel("Epoche")
plt.tight_layout(); plt.show()